# ETL - Gold Layer

Esta camada representa a agregação e preparação final dos dados para consumo por usuários finais, dashboards e aplicações. Aqui, os dados são otimizados para queries rápidas e análises.

## Objetivos:
- Agregar dados (somas, médias, contagens).
- Criar métricas de negócio.
- Otimizar para performance (particionamento, caching).
- Preparar para visualizações e relatórios.

In [ ]:
# Importar bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Criar sessão Spark
spark = SparkSession.builder.appName("Gold Layer ETL").getOrCreate()

print("Sessão Spark criada com sucesso!")

In [ ]:
# Ler dados da Silver Layer
silver_df = spark.read.format("delta").load("/tmp/silver/sample_data_clean")

# Mostrar dados da Silver
silver_df.show(5)
print(f"Total de registros na Silver: {silver_df.count()}")

In [ ]:
# Agregações para Gold Layer
# Contagem por cidade
gold_city_count = silver_df.groupBy("cidade").agg(
    count("nome").alias("total_pessoas"),
    avg("idade").alias("idade_media"),
    min("idade").alias("idade_minima"),
    max("idade").alias("idade_maxima")
)

# Contagem por categoria de idade
gold_age_category = silver_df.groupBy("categoria_idade").agg(
    count("nome").alias("total_por_categoria")
)

# Mostrar agregações
gold_city_count.show()
gold_age_category.show()

In [ ]:
# Salvar dados na camada Gold
gold_city_path = "/tmp/gold/city_summary"
gold_age_path = "/tmp/gold/age_category_summary"

gold_city_count.write.format("delta").mode("overwrite").save(gold_city_path)
gold_age_category.write.format("delta").mode("overwrite").save(gold_age_path)

# Criar tabelas Delta
spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_city_summary
USING DELTA
LOCATION '{gold_city_path}'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_age_category_summary
USING DELTA
LOCATION '{gold_age_path}'
""")

print("Dados salvos na camada Gold com sucesso!")

In [ ]:
# Verificar tabelas criadas
print("Tabela gold_city_summary:")
spark.sql("SELECT * FROM gold_city_summary").show()

print("Tabela gold_age_category_summary:")
spark.sql("SELECT * FROM gold_age_category_summary").show()

## Próximos Passos
- Conectar essas tabelas a dashboards no Databricks SQL.
- Configurar refresh automático com Delta Live Tables (DLT).
- Adicionar mais métricas conforme necessidades de negócio.